Before touching paging, writing a tiny plain (non-paged) autoregressive generation loop myself using a small HF model (e.g. Qwen2.5-0.5B) with use_cache=True. Print the shape of past_key_values at each step to see how viscerally the cache grows per token

In [2]:
print("hello world")

hello world


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


c:\research-eng\repro-llm\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:


MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
MAX_NEW_TOKENS = 20

device = "cuda" if torch.cuda.is_available() else "cpu"


In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
).to(device)


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 3684.08it/s]


In [6]:
model.eval

<bound method PreTrainedModel.eval of Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896

In [7]:
messages = [
    {
        "role": "user",
        "content": "Explain what a GPU is in one sentence."
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

In [8]:
text

'<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nExplain what a GPU is in one sentence.<|im_end|>\n<|im_start|>assistant\n'

In [9]:
inputs = tokenizer(
    text,
    return_tensors="pt",
).to(device)

input_ids = inputs["input_ids"]

print("Prompt tokens:", input_ids.shape[-1])


Prompt tokens: 39


In [15]:
past_key_values = None

generated_tokens = []

with torch.no_grad():

    for step in range(MAX_NEW_TOKENS):

        # -----------------------------
        # Prefill
        # -----------------------------
        if past_key_values is None:
            current_input_ids = input_ids

        # -----------------------------
        # Decode
        # -----------------------------
        else:
            current_input_ids = next_token

        outputs = model(
            input_ids=current_input_ids,
            past_key_values=past_key_values,
            use_cache=True,
        )

        # Updated KV cache
        past_key_values = outputs.past_key_values

        # -----------------------------
        # Inspect KV cache
        # -----------------------------

        layer = past_key_values.layers[0]

        key = layer.keys
        value = layer.values
        for layer_idx, layer in enumerate(past_key_values.layers):
            print(
                f"Step {step:02d} | "
                f"input={tuple(current_input_ids.shape)} | "
                f"K={tuple(key.shape)} | "
                f"V={tuple(value.shape)}"
            )

        # -----------------------------
        # Get next token
        # -----------------------------

        logits = outputs.logits[:, -1, :]

        next_token = torch.argmax(
            logits,
            dim=-1,
            keepdim=True,
        )

        generated_tokens.append(next_token.item())

        # -----------------------------
        # EOS
        # -----------------------------

        eos_token_id = tokenizer.eos_token_id

        if next_token.item() == eos_token_id:
            break


output_text = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True,
)

print("\nGenerated:")
print(output_text)

Step 00 | input=(1, 39) | K=(1, 2, 39, 64) | V=(1, 2, 39, 64)
Step 00 | input=(1, 39) | K=(1, 2, 39, 64) | V=(1, 2, 39, 64)
Step 00 | input=(1, 39) | K=(1, 2, 39, 64) | V=(1, 2, 39, 64)
Step 00 | input=(1, 39) | K=(1, 2, 39, 64) | V=(1, 2, 39, 64)
Step 00 | input=(1, 39) | K=(1, 2, 39, 64) | V=(1, 2, 39, 64)
Step 00 | input=(1, 39) | K=(1, 2, 39, 64) | V=(1, 2, 39, 64)
Step 00 | input=(1, 39) | K=(1, 2, 39, 64) | V=(1, 2, 39, 64)
Step 00 | input=(1, 39) | K=(1, 2, 39, 64) | V=(1, 2, 39, 64)
Step 00 | input=(1, 39) | K=(1, 2, 39, 64) | V=(1, 2, 39, 64)
Step 00 | input=(1, 39) | K=(1, 2, 39, 64) | V=(1, 2, 39, 64)
Step 00 | input=(1, 39) | K=(1, 2, 39, 64) | V=(1, 2, 39, 64)
Step 00 | input=(1, 39) | K=(1, 2, 39, 64) | V=(1, 2, 39, 64)
Step 00 | input=(1, 39) | K=(1, 2, 39, 64) | V=(1, 2, 39, 64)
Step 00 | input=(1, 39) | K=(1, 2, 39, 64) | V=(1, 2, 39, 64)
Step 00 | input=(1, 39) | K=(1, 2, 39, 64) | V=(1, 2, 39, 64)
Step 00 | input=(1, 39) | K=(1, 2, 39, 64) | V=(1, 2, 39, 64)
Step 00 